In [12]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, log_loss)
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
import shap

In [15]:
# Load the dataset

# Debug: Check current directory and folder
print(f"Current working directory: {os.getcwd()}")
print(f"Contents of current directory: {os.listdir('.')}")

DATA_FOLDER = 'MachineLearningCVE'
print(f"\nLooking for CSV files in: {os.path.abspath(DATA_FOLDER)}")
print(f"Folder exists: {os.path.exists(DATA_FOLDER)}")

if os.path.exists(DATA_FOLDER):
    print(f"Files in {DATA_FOLDER}: {os.listdir(DATA_FOLDER)}")

csv_files = glob.glob(os.path.join(DATA_FOLDER, '*.csv'))
print(f"\nFound {len(csv_files)} CSV files: {csv_files}\n")

dfs = []
for f in csv_files:
    print(f"  Loading: {os.path.basename(f)}")
    df_temp = pd.read_csv(f, encoding='utf-8', low_memory=False)
    dfs.append(df_temp)
 
if dfs:
    df = pd.concat(dfs, ignore_index=True)
    print(f"\n  Total records loaded: {len(df):,}")
    print(f"  Total features: {df.shape[1]}")
else:
    print("ERROR: No CSV files were loaded!")

Current working directory: c:\Users\ADMIN\Documents\Hybrid-Enhanced-Stack-Ensemble
Contents of current directory: ['.git', '.gitattributes', 'Bagging-DT.ipynb', 'baseline-model.ipynb', 'lightgbm_confusion_matrix.png', 'lightgbm_feature_importance.png', 'lightgbm_model.txt', 'lightgbm_roc_curves.png', 'MachineLearningCVE', 'NetFlow v2 Features.csv', 'NF-BoT-IoT-V2.parquet']

Looking for CSV files in: c:\Users\ADMIN\Documents\Hybrid-Enhanced-Stack-Ensemble\MachineLearningCVE
Folder exists: True
Files in MachineLearningCVE: ['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']

Found 8 CSV files: ['MachineLearningCVE\\Friday-WorkingHours-Afternoon-DDos.pcap_I

In [16]:
# Check Dataset Balance

print("DATASET BALANCE ANALYSIS")

# Get target column
target_col = df.columns[-1]
print(f"\nTarget Column: {target_col}")

# Count class distribution
class_distribution = df[target_col].value_counts().sort_index()
print("\n" + "-"*60)
print("CLASS DISTRIBUTION (Count):")
print("-"*60)
print(class_distribution)

# Calculate percentages
print("\n" + "-"*60)
print("CLASS DISTRIBUTION (%):")
print("-"*60)
class_percentages = (class_distribution / len(df) * 100).round(2)
for cls, pct in class_percentages.items():
    print(f"  Class {cls}: {pct}% ({class_distribution[cls]:,} samples)")

# Calculate balance metrics
print("\n" + "-"*60)
print("BALANCE METRICS:")
print("-"*60)
minority_count = class_distribution.min()
majority_count = class_distribution.max()
balance_ratio = minority_count / majority_count
imbalance_ratio = majority_count / minority_count

print(f"  Minority class: {minority_count:,} samples")
print(f"  Majority class: {majority_count:,} samples")
print(f"  Balance ratio (minority/majority): {balance_ratio:.4f}")
print(f"  Imbalance ratio (majority/minority): {imbalance_ratio:.2f}:1")

# Assessment
print("\n" + "-"*60)
print("BALANCE ASSESSMENT:")
print("-"*60)
if balance_ratio > 0.9:
    print("✓ Dataset is WELL BALANCED")
elif balance_ratio > 0.5:
    print("⚠️ Dataset is MODERATELY BALANCED")
elif balance_ratio > 0.1:
    print("⚠️ Dataset is IMBALANCED")
else:
    print("❌ Dataset is HIGHLY IMBALANCED")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot - Count
class_distribution.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Class Distribution (Count)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Samples')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(class_distribution):
    axes[0].text(i, v + 100, str(v), ha='center', va='bottom', fontweight='bold')

# Pie plot - Percentage
colors = plt.cm.Set3(range(len(class_distribution)))
axes[1].pie(class_distribution, labels=class_distribution.index, autopct='%1.2f%%', 
            startangle=90, colors=colors, textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1].set_title('Class Distribution (%)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*60)

DATASET BALANCE ANALYSIS

Target Column:  Label

------------------------------------------------------------
CLASS DISTRIBUTION (Count):
------------------------------------------------------------
 Label
BENIGN                        2273097
Bot                              1966
DDoS                           128027
DoS GoldenEye                   10293
DoS Hulk                       231073
DoS Slowhttptest                 5499
DoS slowloris                    5796
FTP-Patator                      7938
Heartbleed                         11
Infiltration                       36
PortScan                       158930
SSH-Patator                      5897
Web Attack � Brute Force         1507
Web Attack � Sql Injection         21
Web Attack � XSS                  652
Name: count, dtype: int64

------------------------------------------------------------
CLASS DISTRIBUTION (%):
------------------------------------------------------------
  Class BENIGN: 80.3% (2,273,097 samples)
  Class B

In [17]:
#DATA PREPROCESSING

print("\n" + "=" * 60)
print("STEP 2: Data Preprocessing")
print("=" * 60)
 
# Standardize column names
df.columns = df.columns.str.strip()
 
# Identify label column
label_col = 'Label'
if label_col not in df.columns:
    label_col = [c for c in df.columns if 'label' in c.lower()][0]
print(f"  Label column: '{label_col}'")
 
# Show class distribution
print(f"\n  Class distribution (before cleaning):")
print(df[label_col].value_counts())
 
# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"\n  Duplicates removed: {before - len(df):,}")
 
# Handle missing and infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
missing = df.isnull().sum().sum()
print(f"  Missing/Inf values found: {missing:,}")
df.dropna(inplace=True)
print(f"  Records after cleaning: {len(df):,}")
 
# Separate features and labels
X = df.drop(columns=[label_col])
y = df[label_col]
 
# Keep only numeric features
X = X.select_dtypes(include=[np.number])
print(f"  Numeric features used: {X.shape[1]}")
 
# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"\n  Attack classes ({len(le.classes_)}):")
for i, cls in enumerate(le.classes_):
    count = (y_encoded == i).sum()
    print(f"    [{i}] {cls}: {count:,} samples")
 
# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)


STEP 2: Data Preprocessing
  Label column: 'Label'

  Class distribution (before cleaning):
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

  Duplicates removed: 308,381
  Missing/Inf values found: 3,128
  Records after cleaning: 2,520,798
  Numeric features used: 78

  Attack classes (15):
    [0] BENIGN: 2,095,057 samples
    [1] Bot: 1,948 samples
    [2] DDoS: 128,014 samples
    [3] DoS GoldenEye: 10,286 samples
    [4] DoS Hulk: 172,8